# Step 1: Data Collection and Preparation




###Framework Setup

---



Installed the necessary libraries and colab setup for executing the research\
\
**Libraries used**:
*   pandas
*   numpy
*   sklearn
*   time
*   xgboost









In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier, StackingClassifier, VotingClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
import time
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# You can now access your files using their path in Drive,
# e.g., '/content/drive/MyDrive/YourDataFolder/nla_2017_phytoplankton_count-data.csv'

##Data Processing

---


Water quality data was obtained from the EPA's National Lakes Assessment 2017, which surveyed lakes across the United States. Three datasets were merged:
- Phytoplankton count data - containing cyanobacteria biovolume measurements by algal group
- Water profile data - containing sensor measurements at surface depth (less than or equal to 1.0m): temperature (°C), pH, and conductivity (μS/cm)
- Water chemistry data - containing turbidity measurements (NTU)

**Processing Steps**
- Cyanobacteria percentage was calculated as (cyanobacteria biovolume / total phytoplankton biovolume) × 100 for each lake
- Samples with cyanobacteria percentage >10% were labeled as "bloom" (1), otherwise "no bloom" (0)
- The 10% cyanobacteria biovolume threshold was selected based on EPA guidelines for harmful algal bloom monitoring, with modification to enable early bloom detection. While EPA typically uses higher thresholds for public health advisories, a 10% threshold allows for earlier identification of developing blooms before they reach critical levels.
- Surface measurements (depth of water less than or equal to 1.0 m) were averaged for each lake
- All three datasets were merged by lake UID (unique identifier)
- Incomplete records with missing values were removed\

**Final Dataset Statistics**

- After incomplete records were removed, there were a total of 941 lake samples, with each of the 4 sensor measurements (temperature, pH, conductivity, and turbidity)
- The target variable being predicted is our calculated cyanobacteria bloom percentage being represented as a 0 for no bloom and a 1 for bloom present.
- Class distribution: 453 lakes with blooms (48.1%), 488 lakes without blooms (51.9%)
- The sample was split into 80% training (752 lakes) set, and a 20% testing (189 lakes) set using stratified sampling to maintain class balance





In [ ]:
threshold = 10

print("="*70)
print("LOADING DATA FOR CLASSIFICATION (Bloom Presence Detection)")
print("="*70)

print("\n Loading phytoplankton data...")
phyto = pd.read_csv('/content/drive/MyDrive/nla_2017_phytoplankton_count-data.csv', encoding='latin1')
print(f"   Loaded {len(phyto)} phytoplankton records")

print("\n Calculating cyanobacteria percentage...")

cyano = phyto[phyto['ALGAL_GROUP'] == 'BLUE-GREEN ALGAE']

cyano_biovolume = cyano.groupby('UID')['BIOVOLUME'].sum().reset_index()
cyano_biovolume.columns = ['UID', 'CYANO_BIOVOLUME']

total_biovolume = phyto.groupby('UID')['BIOVOLUME'].sum().reset_index()
total_biovolume.columns = ['UID', 'TOTAL_BIOVOLUME']

cyano_pct = cyano_biovolume.merge(total_biovolume, on='UID', how='right')
cyano_pct['CYANO_BIOVOLUME'] = cyano_pct['CYANO_BIOVOLUME'].fillna(0)
cyano_pct['CYANO_PCT'] = (cyano_pct['CYANO_BIOVOLUME'] / cyano_pct['TOTAL_BIOVOLUME']) * 100

print(f"   Calculated cyano % for {len(cyano_pct)} lakes")
print(f"   Range: {cyano_pct['CYANO_PCT'].min():.2f}% - {cyano_pct['CYANO_PCT'].max():.2f}%")
print(f"   Mean: {cyano_pct['CYANO_PCT'].mean():.2f}%")

print("\n  Loading sensor data...")
profile = pd.read_csv('/content/drive/MyDrive/nla_2017_profile-data.csv', encoding='latin1')

surface = profile[profile['DEPTH'] <= 1.0]

surface['TEMPERATURE'] = pd.to_numeric(surface['TEMPERATURE'], errors='coerce')
surface['PH'] = pd.to_numeric(surface['PH'], errors='coerce')
surface['CONDUCTIVITY'] = pd.to_numeric(surface['CONDUCTIVITY'], errors='coerce')

sensors = surface.groupby('UID').agg({
    'TEMPERATURE': 'mean',
    'PH': 'mean',
    'CONDUCTIVITY': 'mean'
}).reset_index()

print(f"   Sensor data for {len(sensors)} lakes")

# ============================================
# NEW: Load turbidity data
# ============================================
print("\n Loading turbidity data...")
chem = pd.read_csv('/content/drive/MyDrive/nla_2017_water_chemistry_chla-data.csv', encoding='latin1')

# Filter for turbidity measurements
turb_data = chem[chem['ANALYTE'] == 'TURB'].copy()

# Convert RESULT to numeric
turb_data['RESULT'] = pd.to_numeric(turb_data['RESULT'], errors='coerce')

# Get turbidity for each UID (average if multiple measurements)
turbidity = turb_data.groupby('UID')['RESULT'].mean().reset_index()
turbidity.columns = ['UID', 'TURB']

print(f"   Turbidity data for {len(turbidity)} lakes")
print(f"   Range: {turbidity['TURB'].min():.2f} - {turbidity['TURB'].max():.2f} NTU")
print(f"   Mean: {turbidity['TURB'].mean():.2f} NTU")

# ============================================
# Merge everything
# ============================================
print("\n Merging datasets...")
final_data = cyano_pct[['UID', 'CYANO_PCT']].merge(sensors, on='UID', how='inner')

# Add turbidity
print("   Adding turbidity...")
final_data = final_data.merge(turbidity, on='UID', how='inner')

print(f"   Before removing nulls: {len(final_data)} lakes")
final_data = final_data.dropna()
print(f"   After removing nulls: {len(final_data)} lakes")

print(f"\n Creating classification target (threshold = {threshold}%)...")

final_data['CYANO_BLOOM'] = (final_data['CYANO_PCT'] > threshold).astype(int)

no_bloom = (final_data['CYANO_BLOOM'] == 0).sum()
bloom = (final_data['CYANO_BLOOM'] == 1).sum()

print(f"\n   Class Distribution:")
print(f"   ├─ No Bloom (≤{threshold}%):  {no_bloom} lakes ({no_bloom/len(final_data)*100:.1f}%)")
print(f"   └─ Bloom (>{threshold}%):     {bloom} lakes ({bloom/len(final_data)*100:.1f}%)")

balance_ratio = min(no_bloom, bloom) / max(no_bloom, bloom)
if balance_ratio < 0.3:
    print(f"\n   WARNING: Classes are imbalanced (ratio = {balance_ratio:.2f})")
    print(f"   Consider adjusting threshold or using class weights in models")
else:
    print(f"\n   Classes are reasonably balanced (ratio = {balance_ratio:.2f})")

print("\n" + "="*70)
print(" FINAL DATASET SUMMARY")
print("="*70)
print(f"Total lakes: {len(final_data)}")
print(f"\nFeatures (Input):")
print(f"  1. TEMPERATURE (°C)")
print(f"  2. PH")
print(f"  3. CONDUCTIVITY (μS/cm)")
print(f"  4. TURB (NTU)")
print(f"\nTarget (Output):")
print(f"  CYANO_BLOOM: 0 = No Bloom, 1 = Bloom")
print(f"\nCyano % kept for reference (not used in training)")

print("\n" + "="*70)
print("DATASET PREVIEW")
print("="*70)
print(final_data.head(10))

print("\n" + "="*70)
print("STATISTICAL SUMMARY")
print("="*70)
print(final_data.describe())

# Check correlation between turbidity and cyano
print("\n" + "="*70)
print("TURBIDITY vs CYANOBACTERIA CORRELATION")
print("="*70)
turb_corr = final_data[['CYANO_PCT', 'TURB', 'CYANO_BLOOM']].corr()
print(turb_corr)
print(f"\nTurbidity correlation with Cyano%: {turb_corr.loc['CYANO_PCT', 'TURB']:.3f}")

filename = f'cyano_classification_data_threshold_{threshold}_with_turb.csv'
final_data.to_csv(f'/content/drive/MyDrive/{filename}', index=False)
print(f"\n Saved to: /content/drive/MyDrive/{filename}")

# Step 2: Inital Model Training and Optimization

##Basline Model Training

---


**Algorithm Selection:**
Five machine learning algorithms were selected from the sckit-learn machine learning library to represent diverse methodologies:
- Logistic Regression
  -  linear baseline model
- Random Forest
  - tree-based ensemble method
- Gradient Boosting
  - sequential boosting algorithm
- XGBoost
  - optimized gradient boosting implementation
- Multi-Layer Perceptron (MLP)
  - neural network


**Each model was trained on the dataset and evaluated using the default parameters**


In [ ]:
model_results = []

In [ ]:
print("="*70)
print("TRAINING CLASSIFICATION MODELS FOR CYANOBACTERIA BLOOM DETECTION")
print("="*70)

data = pd.read_csv('/content/drive/MyDrive/cyano_classification_data_threshold_10_with_turb.csv')

threshold = 10
data['CYANO_BLOOM'] = (data['CYANO_PCT'] > threshold).astype(int)

print("="*60)
print(f"CLASSIFICATION: Bloom Threshold = {threshold}%")
print("="*60)
print(f"Total lakes: {len(data)}")
print(f"No bloom (≤{threshold}%): {(data['CYANO_BLOOM']==0).sum()} ({(data['CYANO_BLOOM']==0).sum()/len(data)*100:.1f}%)")
print(f"Bloom (>{threshold}%): {(data['CYANO_BLOOM']==1).sum()} ({(data['CYANO_BLOOM']==1).sum()/len(data)*100:.1f}%)")

X = data[['TEMPERATURE', 'PH', 'CONDUCTIVITY', "TURB"]]
y = data['CYANO_BLOOM']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining set: {len(X_train)} lakes")
print(f"Test set: {len(X_test)} lakes")

results = []

print("\n" + "="*60)
print("Training Logistic Regression...")
start = time.time()
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
train_time = time.time()-start

y_pred = lr_model.predict(X_test)
results.append({
    'Model': 'Logistic Regression',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'Training_Time': train_time,
    'ROC_AUC': roc_auc_score(y_test, y_pred)
})
print(f"Done in {train_time:.2f} seconds")

print("/n" +"="*60)
print("Training Xgboost Classifier...")
start = time.time()
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
train_time = time.time()-start
y_pred = xgb.predict(X_test)
results.append({
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'Training_Time': train_time,
    'ROC_AUC': roc_auc_score(y_test, y_pred)
})
print("\n" + "="*60)
print("Training Random Forest Classifier...")
start = time.time()
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
train_time = time.time()-start

y_pred = rf_model.predict(X_test)
results.append({
    'Model': 'Random Forest',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'Training_Time': train_time,
    'ROC_AUC': roc_auc_score(y_test, y_pred)
})
print(f"Done in {train_time:.2f} seconds")

print("\n" + "="*60)
print("Training Gradient Boosting Classifier...")
start = time.time()
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
train_time = time.time()-start

y_pred = gb_model.predict(X_test)
results.append({
    'Model': 'Gradient Boosting',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'Training_Time': train_time,
    'ROC_AUC': roc_auc_score(y_test, y_pred)
})
print(f"Done in {train_time:.2f} seconds")

print("\n" + "="*60)
print("Training Neural Network Classifier...")
start = time.time()
nn_model = MLPClassifier(hidden_layer_sizes=(64, 32), random_state=42, max_iter=500)
nn_model.fit(X_train, y_train)
train_time = time.time()-start

y_pred = nn_model.predict(X_test)
results.append({
    'Model': 'Neural Network',
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'Training_Time': train_time,
    'ROC_AUC': roc_auc_score(y_test, y_pred)
})
print(f"Done in {train_time:.2f} seconds")

print("\n\n" + "="*80)
print("📊 SUMMARY - CLASSIFICATION MODEL COMPARISON")
print("="*80 + "\n")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('ROC_AUC', ascending=False) # Changed sorting to ROC_AUC

print(f"{'Model':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Time (s)':<10} {'ROC_AUC':<10}") # Added ROC_AUC to header
print("-"*100) # Adjusted separator length
for _, row in results_df.iterrows():
    print(f"{row['Model']:<25} {row['Accuracy']:<12.4f} {row['Precision']:<12.4f} {row['Recall']:<12.4f} {row['F1-Score']:<12.4f} {row['Training_Time']:<10.2f} {row['ROC_AUC']:<10.4f}") # Added ROC_AUC to row printing

print("\n" + "="*80)
print("🏆 BEST MODEL")
print("="*80)
best = results_df.iloc[0]
print(f"Model: {best['Model']}")
print(f"Accuracy: {best['Accuracy']:.2%} (correct predictions)")
print(f"Precision: {best['Precision']:.2%} (when it says 'bloom', it's right {best['Precision']:.0%} of the time)")
print(f"Recall: {best['Recall']:.2%} (catches {best['Recall']:.0%} of actual blooms)")
print(f"F1-Score: {best['F1-Score']:.4f} (overall balance)")
print(f"ROC AUC: {best['ROC_AUC']:.4f} (discriminative power)") # Added ROC_AUC for best model

print("\n" + "="*80)
print("💡 INTERPRETATION FOR ROV")
print("="*80)
print(f"With 4 sensors, the {best['Model']} can detect blooms with {best['Accuracy']:.0%} accuracy.")
print(f"This means your ROV will correctly identify bloom conditions {best['Accuracy']*100:.0f} out of 100 times.")
print("="*80 + "\n")

print("\n" + "="*80)
print("CONFUSION MATRIX (Best Model)")
print("="*80)
best_model_name = best['Model']
if best_model_name == 'Random Forest':
    best_model = rf_model
elif best_model_name == 'Logistic Regression':
    best_model = lr_model
elif best_model_name == 'Gradient Boosting':
    best_model = gb_model
else:
    best_model = nn_model

y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
model_results.append(results)



print("\n                Predicted")
print("              No Bloom  Bloom")
print(f"Actual No Bloom   {cm[0,0]:<6}  {cm[0,1]:<6}")
print(f"       Bloom      {cm[1,0]:<6}  {cm[1,1]:<6}")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=['No Bloom', 'Bloom']))

In [ ]:
print(model_results)

## Hyperparameter Tuning

---

**Hyperparameter tuning techniques were used in order to maximize the effectiveness of each algorithm**
- Hyperparameters are settings that control how a machine learning model learns and are set before training
- Hyperparameter tuning is the process of testing different combinations of these settings to find the version that produces the best performance for our desired results.

**In this research we used two tuning techniques (imported from sckit-learn) GridSearchCV and Randomized SearchCV**
- GridSearchCV is an exhaustive search which tests all possible combinations and was used for tuning our Logistic Regression and MLP algorithms
- RandomizedSearchCV was used for Random Forest, Gradient Boosting, and XGBoost (randomly samples 100 combinations from the parameter space, more efficient for large search spaces)

**All models employed 5-fold cross-validation**
- The training data is split into 5 equal parts
- The model trains on 4 parts and tests on the remaining 1 part
- This process repeats 5 times, with each part serving as the test set once
- Final performance is the average across all 5 iterations, ensuring the model performs consistently

**Each algorithm was tuned to be optimized for the highest ROC-AUC score**
- ROC-AUC (Receiver Operating Characteristic - Area Under Curve) measures how well the model distinguishes between the two classes (bloom vs. no bloom)
  - scores closer to 1.0 indicating better performance
- It was selected as the primary evaluation metric because it is robust to class imbalance biases, considers all possible decision thresholds, and it is widely used in water quality prediction and environmental monitoring

**Decision Threshold Optimization**
- The default prediction threshold of 0.5 was evaluated and adjusted to prioritize public health safety
- A threshold of 0.35 was selected for final predictions
- In water quality monitoring, missing an actual bloom (false negative) is more dangerous than issuing an unnecessary warning (false positive)
- This lower threshold increases recall (catching more blooms) while accepting slightly lower precision (more false alarms)
- ROC-AUC scores remain unchanged as they are threshold-independent



###Random Forest Hyperparameter tuning optimization
Hyperparameters:
- number of trees (n_estimators)
- maximum tree depth
- minimum samples required to split nodes
- minimum samples per leaf
- bootstrap sampling


In [ ]:
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start=100, stop=500, num=10)],
    'max_features': ['sqrt', 'log2', None],  # Now actually tuning this
    'max_depth': [int(x) for x in np.linspace(10, 110, num=11)] + [None],  # Added None option
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

rf = RandomForestClassifier(random_state=42)

rf_grid = RandomizedSearchCV(
    estimator= rf,
    param_distributions=param_dist,
    n_iter=100,
    cv=5,
    verbose=2,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_

# Get probabilities
rf_test_probs = best_rf.predict_proba(X_test)[:, 1]

# Use custom threshold of 0.35 instead of default 0.5
optimal_threshold = 0.35
rf_test_preds = (rf_test_probs >= optimal_threshold).astype(int)

print(f"Using threshold: {optimal_threshold}")
print("Accuracy:", accuracy_score(y_test, rf_test_preds))
print("Recall:", recall_score(y_test, rf_test_preds))
print("F1:", f1_score(y_test, rf_test_preds))
print("ROC-AUC:", roc_auc_score(y_test, rf_test_probs))  # Using probabilities (threshold-independent)

model_results.append({
    'Model': 'rf_optimized',
    'Accuracy': accuracy_score(y_test, rf_test_preds),
    'Precision': precision_score(y_test, rf_test_preds),
    'Recall': recall_score(y_test,rf_test_preds),
    'F1-Score': f1_score(y_test, rf_test_preds),
    'ROC_AUC': roc_auc_score(y_test, rf_test_probs)
})

###Gradient Boosting Hyperparameter tuning optimization
Hyperparameters:
- learning rate (controls contribution of each tree)
- maximum tree depth
- subsample ratio (fraction of data per tree)
- minimum samples for splits and leaves


In [ ]:
gb = GradientBoostingClassifier(random_state=42)

gb_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5, 7],
    "subsample": [0.8, 0.9, 1.0],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

gb_grid = RandomizedSearchCV(
    gb,
    gb_param_dist,
    n_iter=100,  # Sample 100 combinations
    cv=5,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

gb_grid.fit(X_train, y_train)
best_gb = gb_grid.best_estimator_
print(f'Best Gradient Boosting Model: {best_gb}')

# Get probabilities
gb_test_probs = best_gb.predict_proba(X_test)[:, 1]

# Use custom threshold of 0.35 instead of default 0.5
optimal_threshold = 0.35
gb_test_preds = (gb_test_probs >= optimal_threshold).astype(int)

print(f"Using threshold: {optimal_threshold}")
print("Accuracy:", accuracy_score(y_test, gb_test_preds))
print("Recall:", recall_score(y_test, gb_test_preds))
print("F1:", f1_score(y_test, gb_test_preds))
print("ROC-AUC:", roc_auc_score(y_test, gb_test_probs))
#
model_results.append({
    'Model': 'gb_optimized',
    'Accuracy': accuracy_score(y_test, gb_test_preds),
    'Precision': precision_score(y_test, gb_test_preds),
    'Recall': recall_score(y_test,gb_test_preds),
    'F1-Score': f1_score(y_test, gb_test_preds),
    'ROC_AUC': roc_auc_score(y_test, gb_test_probs)
})

###XGBoost Hyperparameter Tuning Optimization
Hyperparameters:
- learning rate
- maximum depth
- subsample ratio
- column sampling ratio (fraction of features used per tree)


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=42, eval_metric='logloss')

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

xgb_grid = RandomizedSearchCV(xgb, xgb_params, n_iter=50, cv=5,
                               scoring='roc_auc', random_state=42, n_jobs=-1)
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_

xgb_probs = best_xgb.predict_proba(X_test)[:, 1]
optimal_threshold = 0.35
xgb_test_preds = (xgb_probs >= optimal_threshold).astype(int)

print(f"XGBoost - Using threshold: {optimal_threshold}")
print("Accuracy:", accuracy_score(y_test, xgb_test_preds))
print("Recall:", recall_score(y_test, xgb_test_preds))
print("F1:", f1_score(y_test, xgb_test_preds))
print("ROC-AUC:", roc_auc_score(y_test, xgb_probs))

model_results.append({
    'Model': 'xgb_optimized',
    'Accuracy': accuracy_score(y_test, xgb_test_preds),
    'Precision': precision_score(y_test, xgb_test_preds),
    'Recall': recall_score(y_test,xgb_test_preds),
    'F1-Score': f1_score(y_test, xgb_test_preds),
    'ROC_AUC': roc_auc_score(y_test, xgb_probs)
})

###Logistic Regression Hyperparameter Tuning Optimization
Hyperparameters:
- regularization strength (C: controls overfitting)
- penalty type (L1/L2: different approaches to prevent overfitting)


In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)

lr_param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],  # More regularization options
    "penalty": ['l1', 'l2'],  # Different regularization types
    "solver": ['liblinear', 'saga']  # Solvers that support both l1 and l2
}

lr_grid = GridSearchCV(
    lr,
    lr_param_grid,
    cv=5,
    scoring="roc_auc",  # Match your evaluation metric
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train, y_train)
best_lr = lr_grid.best_estimator_
print(f'Best Logistic Regression Model: {best_lr}')

lr_test_probs = best_lr.predict_proba(X_test)[:, 1]
lr_test_preds = (lr_test_probs >= optimal_threshold).astype(int)

print(f"Logistic Regression - Using threshold: {optimal_threshold}")
print("Accuracy:", accuracy_score(y_test, lr_test_preds))
print("Recall:", recall_score(y_test, lr_test_preds))
print("F1:", f1_score(y_test, lr_test_preds))
print("ROC-AUC:", roc_auc_score(y_test, lr_test_probs))

model_results.append({
    'Model': 'lr_optimized',
    'Accuracy': accuracy_score(y_test, lr_test_preds),
    'Precision': precision_score(y_test, lr_test_preds),
    'Recall': recall_score(y_test,lr_test_preds),
    'F1-Score': f1_score(y_test, lr_test_preds),
    'ROC_AUC': roc_auc_score(y_test, lr_test_probs)
})

###MLP Hyperparameter tuning optimization
Data was first standardized using StandardScaler pipeline (normalizes feature ranges)

Hyperparameters:
- hidden layer architecture (network size and depth)
- activation functions (relu/tanh)
- regularization parameter (alpha)
- initial learning rate


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Pipeline = scaling + MLP
mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        random_state=42,
        early_stopping=True,
        max_iter=1000
    ))
])

mlp_param_grid = {
    "mlp__hidden_layer_sizes": [(32,), (64,), (128,), (64, 32), (128, 64)],  # More options
    "mlp__activation": ['relu', 'tanh'],  # Different activation functions
    "mlp__alpha": [0.0001, 0.001, 0.01],
    "mlp__learning_rate_init": [0.001, 0.01]
}

mlp_grid = RandomizedSearchCV(
    mlp_pipe,
    mlp_param_grid,
    n_iter=50,
    cv=5,
    scoring="roc_auc",  # Match your evaluation metric
    n_jobs=-1,
    verbose=1
)

mlp_grid.fit(X_train, y_train)
best_mlp = mlp_grid.best_estimator_
print(f'Best MLP Model: {best_mlp}')

mlp_test_probs = best_mlp.predict_proba(X_test)[:, 1]
mlp_test_preds = (mlp_test_probs >= optimal_threshold).astype(int)

print(f"MLP - Using threshold: {optimal_threshold}")
print("Accuracy:", accuracy_score(y_test, mlp_test_preds))
print("Recall:", recall_score(y_test, mlp_test_preds))
print("F1:", f1_score(y_test, mlp_test_preds))
print("ROC-AUC:", roc_auc_score(y_test, mlp_test_probs))

model_results.append({
    'Model': 'mlp_optimized',
    'Accuracy': accuracy_score(y_test, mlp_test_preds),
    'Precision': precision_score(y_test, mlp_test_preds),
    'Recall': recall_score(y_test,mlp_test_preds),
    'F1-Score': f1_score(y_test, mlp_test_preds),
    'ROC_AUC': roc_auc_score(y_test, mlp_test_probs)
})





#Step 3: Feauture Engineering

##New Inputs

---


- Feature engineering is the process of creating new variables (features) from existing data to help machine learning models detect patterns more easily
- While raw sensor readings provide valuable information, combining and transforming them based on scientific knowledge of cyanobacteria biology can reveal relationships that models might otherwise miss
- Nine engineered features were created from the original five sensor measurements based on previous research on bloom formation

**Engineered Features:**
- WARM_WATER
  - flags when temperature >20°C
  - Cyanobacteria blooms grow best in warm water between 20-30°C (Singh 2015)
- HIGH_PH:
  - flags when pH >8.5
  - When cyanobacteria photosynthesize, they consume CO2 which raises the water's pH to 8.5-10 (EPA, 2019)
- HIGH_CONDUCTIVITY:
  - flags when conductivity is in the top 25% of all samples
  - High conductivity means more dissolved minerals in the water, which indicates nutrient-rich conditions that blooms need to grow
- PH_DEVIATION:
  - measures distance from neutral pH (7.0)
  - Shows how far pH is from normal (7.0); larger values indicate unusual water chemistry
- TEMP_PH_INTERACTION:
  - temperature multiplied by pH
  - Combines warm water and high pH; together they indicate active blooms more strongly than either measurement alone
- TEMP_COND_INTERACTION:
  - temperature multiplied by conductivity
  - Combines warm water with high nutrients; this combination creates ideal conditions for blooms



###Engineered Dataset

---


A new dataset was created off of the old dataset, the only difference being the addition of the new input features

In [ ]:

# Load data
df = pd.read_csv('/content/drive/MyDrive/cyano_classification_data_threshold_10_with_turb.csv')

# Create new features
df['WARM_WATER'] = (df['TEMPERATURE'] > 20).astype(int)
df['HIGH_PH'] = (df['PH'] > 8.5).astype(int)
df['PH_DEVIATION'] = abs(df['PH'] - 7.0)
df['HIGH_CONDUCTIVITY'] = (df['CONDUCTIVITY'] > df['CONDUCTIVITY'].quantile(0.75)).astype(int)
df['TEMP_PH_INTERACTION'] = df['TEMPERATURE'] * df['PH']
df['TEMP_COND_INTERACTION'] = df['TEMPERATURE'] * df['CONDUCTIVITY']


# Save
df.to_csv('/content/drive/MyDrive/cyano_engineered.csv', index=False)
print(f" Saved! Total features: {df.shape[1]}")

In [ ]:

# Load engineered data
df = pd.read_csv('/content/drive/MyDrive/cyano_engineered.csv')

# Prepare X and y
X = df.drop(['UID', 'CYANO_PCT', 'CYANO_BLOOM'], axis=1)
y = df['CYANO_BLOOM']

# Split
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Training: {len(X_train)}, Testing: {len(X_test)}")

##Model Training (with engineered features)

---


All five machine learning models were retrained on the expanded dataset
- 11 total features (5 original + 6 engineered)
The same hyperparameter tuning process from Step 2 was repeated:
- GridSearchCV for Logistic Regression and MLP
- RandomizedSearchCV (100 iterations) for Random Forest, Gradient Boosting, and XGBoost
- 5-fold cross-validation for all models
- Optimization for ROC-AUC score




Performance was compared between models trained on original features (Step 2) versus engineered features to measure the impact of feature engineering on prediction accuracy


###Logistic Regression Training

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)

lr_params = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ['l1', 'l2'],
    "solver": ['liblinear', 'saga']
}

lr_grid = GridSearchCV(lr, lr_params, cv=5, scoring="roc_auc", n_jobs=-1)
lr_grid.fit(X_train_eng, y_train_eng)
best_lr = lr_grid.best_estimator_
lr_probs = best_lr.predict_proba(X_test_eng)[:, 1]
lr_preds = (lr_probs >= optimal_threshold).astype(int)
print("Logistic Regression:")
print(f"  Accuracy:  {accuracy_score(y_test_eng, lr_preds):.4f}")
print(f"  Recall:    {recall_score(y_test_eng, lr_preds):.4f}")
print(f"  F1:        {f1_score(y_test_eng, lr_preds):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test_eng, lr_probs):.4f}")

model_results.append({
    'Model': 'lr_features',
    'Accuracy': accuracy_score(y_test, lr_preds),
    'Precision': precision_score(y_test, lr_preds),
    'Recall': recall_score(y_test,lr_preds),
    'F1-Score': f1_score(y_test, lr_preds),
    'ROC_AUC': roc_auc_score(y_test, lr_probs)
})

###Random Forest Training

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf_params = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_features': ['sqrt', 'log2', None],
    'max_depth': [10, 30, 50, 70, 90, 110, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

rf_grid = RandomizedSearchCV(rf, rf_params, n_iter=100, cv=5, scoring="roc_auc", random_state=42, n_jobs=-1, verbose=2)
rf_grid.fit(X_train_eng, y_train_eng)
best_rf = rf_grid.best_estimator_
rf_probs = best_rf.predict_proba(X_test_eng)[:, 1]
rf_preds = (rf_probs >= optimal_threshold).astype(int)
print("Random Forest:")
print(f"  Accuracy:  {accuracy_score(y_test_eng, rf_preds):.4f}")
print(f"  Recall:    {recall_score(y_test_eng, rf_preds):.4f}")
print(f"  F1:        {f1_score(y_test_eng, rf_preds):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test_eng, rf_probs):.4f}")

model_results.append({
    'Model': 'rf_features',
    'Accuracy': accuracy_score(y_test, rf_preds),
    'Precision': precision_score(y_test, rf_preds),
    'Recall': recall_score(y_test,rf_preds),
    'F1-Score': f1_score(y_test, rf_preds),
    'ROC_AUC': roc_auc_score(y_test, rf_probs)
})

###Gradient Boosting Training

In [ ]:
gb = GradientBoostingClassifier(random_state=42)

gb_params = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5, 7],
    "subsample": [0.8, 0.9, 1.0],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

gb_grid = RandomizedSearchCV(gb, gb_params, n_iter=100, cv=5, scoring="roc_auc", random_state=42, n_jobs=-1, verbose=2)
gb_grid.fit(X_train_eng, y_train_eng)
best_gb = gb_grid.best_estimator_
gb_probs = best_gb.predict_proba(X_test_eng)[:, 1]
gb_preds = (gb_probs >= optimal_threshold).astype(int)
print("Gradient Boosting:")
print(f"  Accuracy:  {accuracy_score(y_test_eng, gb_preds):.4f}")
print(f"  Recall:    {recall_score(y_test_eng, gb_preds):.4f}")
print(f"  F1:        {f1_score(y_test_eng, gb_preds):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test_eng, gb_probs):.4f}")

model_results.append({
    'Model': 'gb_features',
    'Accuracy': accuracy_score(y_test, gb_preds),
    'Precision': precision_score(y_test, gb_preds),
    'Recall': recall_score(y_test,gb_preds),
    'F1-Score': f1_score(y_test, gb_preds),
    'ROC_AUC': roc_auc_score(y_test, gb_probs)
})

###MLP Training

In [ ]:
mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(random_state=42, early_stopping=True, max_iter=1000))
])

mlp_params = {
    "mlp__hidden_layer_sizes": [(32,), (64,), (128,), (64, 32), (128, 64)],
    "mlp__activation": ['relu', 'tanh'],
    "mlp__alpha": [0.0001, 0.001, 0.01],
    "mlp__learning_rate_init": [0.001, 0.01]
}

mlp_grid = GridSearchCV(mlp_pipe, mlp_params, cv=5, scoring="roc_auc", n_jobs=-1, verbose=1)
mlp_grid.fit(X_train_eng, y_train_eng)
best_mlp = mlp_grid.best_estimator_
mlp_probs = best_mlp.predict_proba(X_test_eng)[:, 1]
mlp_preds = (mlp_probs >= optimal_threshold).astype(int)
print("MLP:")
print(f"  Accuracy:  {accuracy_score(y_test_eng, mlp_preds):.4f}")
print(f"  Recall:    {recall_score(y_test_eng, mlp_preds):.4f}")
print(f"  F1:        {f1_score(y_test_eng, mlp_preds):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test_eng, mlp_probs):.4f}")

model_results.append({
    'Model': 'mlp_features',
    'Accuracy': accuracy_score(y_test, mlp_preds),
    'Precision': precision_score(y_test, mlp_preds),
    'Recall': recall_score(y_test, mlp_preds),
    'F1-Score': f1_score(y_test, mlp_preds),
    'ROC_AUC': roc_auc_score(y_test, mlp_probs)
})

###XGB Training

In [ ]:
xgb = XGBClassifier(random_state=42, eval_metric='logloss')

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

xgb_grid = RandomizedSearchCV(xgb, xgb_params, n_iter=50, cv=5,
                               scoring='roc_auc', random_state=42, n_jobs=-1)
xgb_grid.fit(X_train_eng, y_train_eng)
best_xgb = xgb_grid.best_estimator_
xgb_probs = best_xgb.predict_proba(X_test_eng)[:, 1]
xgb_preds = (xgb_probs >= optimal_threshold).astype(int)
print("XGBoost:")
print(f"  Accuracy:  {accuracy_score(y_test_eng, xgb_preds):.4f}")
print(f"  Recall:    {recall_score(y_test_eng, xgb_preds):.4f}")
print(f"  F1:        {f1_score(y_test_eng, xgb_preds):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test_eng, xgb_probs):.4f}")

model_results.append({
    'Model': 'xgb_features',
    'Accuracy': accuracy_score(y_test, xgb_preds),
    'Precision': precision_score(y_test, xgb_preds),
    'Recall': recall_score(y_test,xgb_preds),
    'F1-Score': f1_score(y_test, xgb_preds),
    'ROC_AUC': roc_auc_score(y_test, xgb_probs)
})

#Step 4: Model Comparison and Ensemble Methods


##Ensemble Methods

---


We tested two types of Ensemble methods to combine predictions from multiple models to potentially achieve better performance than any single model
- Stacking Ensemble Classifier
  - Uses predictions from multiple "base models" as inputs to a "meta-learner" model
  - Various combinations of base models were tested (ie. MLP + Gradient Boosting, all five models together)
  - Logistic Regression was used as the meta-learner to combine base model predictions
  - 5-fold cross-validation was used during stacking training to prevent overfitting
- Soft Voting Ensemble Classifier:
  - Averages the predicted probabilities from multiple models
  - Various combinations of base models were also tested for this method

Ensemble performance was compared to individual models to determine if combining models provided meaningful improvement


###Stacking Ensemble

In [ ]:
from itertools import combinations

# Your trained models
models = {
    'lr': best_lr,
    'rf': best_rf,
    'gb': best_gb,
    'mlp': best_mlp,
    'xgb': best_xgb  # if you trained XGBoost
}

results = []

# Try all combinations (2, 3, 4, 5 models)
for r in range(2, len(models) + 1):
    for combo in combinations(models.keys(), r):
        # Create estimator list
        estimators = [(name, models[name]) for name in combo]

        # Stacking Classifier
        stack = StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(C=1.0, random_state=42),
            cv=5,
            stack_method="predict_proba",
            n_jobs=-1
        )

        optimal_threshold = 0.35

        # Stacking
        stack.fit(X_train_eng, y_train_eng)
        probs = stack.predict_proba(X_test_eng)[:, 1]
        preds = (probs >= optimal_threshold).astype(int)
        roc = roc_auc_score(y_test_eng, probs)
        recall = recall_score(y_test_eng, preds)

        results.append({
            'Models': ' + '.join(combo),
            'Count': len(combo),
            'ROC-AUC': roc,
            'Recall': recall
        })

        print(f"Tested: {' + '.join(combo):30s} ROC-AUC: {roc:.4f} Recall:{recall:.4f}")

# Sort and display
import pandas as pd
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)

print("\n" + "="*70)
print("TOP 10 ENSEMBLE COMBINATIONS")
print("="*70)
print(results_df.head(10).to_string(index=False))

print(f"\n🏆 BEST COMBINATION:")
best = results_df.iloc[0]
print(f"   Models: {best['Models']}")
print(f"   ROC-AUC: {best['ROC-AUC']:.4f}")



###Voting Ensemble

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from itertools import combinations
import pandas as pd

# Your trained models
models = {
    'lr': best_lr,
    'rf': best_rf,
    'gb': best_gb,
    'mlp': best_mlp,
    'xgb': best_xgb
}

voting_results = []

# Try all combinations (2, 3, 4, 5 models)
for r in range(2, len(models) + 1):
    for combo in combinations(models.keys(), r):
        # Create estimator list
        estimators = [(name, models[name]) for name in combo]

        # Voting Classifier
        voting = VotingClassifier(
            estimators=estimators,
            voting='soft',
            n_jobs=-1
        )

        optimal_threshold=0.35
        voting.fit(X_train_eng, y_train_eng)
        probs = voting.predict_proba(X_test_eng)[:, 1]
        preds = (probs >= optimal_threshold).astype(int)
        roc = roc_auc_score(y_test_eng, probs)
        recall = recall_score(y_test_eng, preds)

        voting_results.append({
            'Models': ' + '.join(combo),
            'Count': len(combo),
            'ROC-AUC': roc,
            'Recall': recall
        })

        print(f"Tested: {' + '.join(combo):30s} ROC-AUC: {roc:.4f} Recall: {recall:.4f}")

# Sort and display
voting_df = pd.DataFrame(voting_results).sort_values('ROC-AUC', ascending=False)

print("\n" + "="*70)
print("TOP 10 VOTING CLASSIFIER COMBINATIONS")
print("="*70)
print(voting_df.head(10).to_string(index=False))

print(f"\n BEST VOTING COMBINATION:")
best_voting = voting_df.iloc[0]
print(f"   Models: {best_voting['Models']}")
print(f"   ROC-AUC: {best_voting['ROC-AUC']:.4f}")


###Ensemble Method Combination Comparison

In [ ]:
from itertools import combinations

# Your trained models
models = {
    'lr': best_lr,
    'rf': best_rf,
    'gb': best_gb,
    'mlp': best_mlp,
    'xgb': best_xgb  # if you trained XGBoost
}

results = []

# Try all combinations (2, 3, 4, 5 models)
for r in range(2, len(models) + 1):
    for combo in combinations(models.keys(), r):
        # Create estimator list
        estimators = [(name, models[name]) for name in combo]

        # Stacking Classifier
        stack = StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(C=1.0, random_state=42),
            cv=5,
            stack_method="predict_proba",
            n_jobs=-1
        )

        optimal_threshold = 0.35

        # Stacking
        stack.fit(X_train_eng, y_train_eng)
        probs = stack.predict_proba(X_test_eng)[:, 1]
        preds = (probs >= optimal_threshold).astype(int)
        roc = roc_auc_score(y_test_eng, probs)
        recall = recall_score(y_test_eng, preds)

        results.append({
            'Models': ' + '.join(combo),
            'Count': len(combo),
            'ROC-AUC': roc,
            'Recall': recall
        })

        print(f"Tested: {' + '.join(combo):30s} ROC-AUC: {roc:.4f} Recall:{recall:.4f}")

# Sort and display
import pandas as pd
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)

print("\n" + "="*70)
print("TOP 10 ENSEMBLE COMBINATIONS")
print("="*70)
print(results_df.head(10).to_string(index=False))

print(f"\n BEST COMBINATION:")
best = results_df.iloc[0]
print(f"   Models: {best['Models']}")
print(f"   ROC-AUC: {best['ROC-AUC']:.4f}")


In [ ]:
print(model_results)

##Evaluation

---


- All models were evaluated on the held-out test set (189 lakes, 20% of total data)
- Each model made predictions on the test set, and multiple performance metrics were calculated

Performance Metrics Compared:
- ROC-AUC (primary metric)
  - measures how well the model ranks bloom samples above non-bloom samples (closer to 1.0 is greater performance)
- Accuracy
  -  percentage of correct predictions (both blooms and non-blooms)
- Recall (secondary metric)
  - of all actual blooms, what percentage did the model correctly identify (measures missed blooms)
- F1-Score
  - balances precision and recall into a single metric (useful when both false alarms and missed blooms matter)

Models were ranked by ROC-AUC to identify the best-performing algorithm
Results were also compared between models trained on original features versus engineered features to measure feature engineering impact


### Feature Importance ranking
Visualizes which features matter most

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Train Random Forest on engineered features
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_eng, y_train_eng)

# Get feature importances
importance_df = pd.DataFrame({
    'Feature': X_train_eng.columns,
    'Importance Score': rf.feature_importances_
}).sort_values('Importance Score', ascending=False)

# Add rank
importance_df['Rank'] = range(1, len(importance_df) + 1)

# Add type (Original vs Engineered)
original_features = ['TEMPERATURE', 'PH', 'OXYGEN', 'CONDUCTIVITY', 'TURB']
importance_df['Type'] = importance_df['Feature'].apply(
    lambda x: 'Original' if x in original_features else 'Engineered'
)

# Reorder columns
importance_df = importance_df[['Rank', 'Feature', 'Importance Score', 'Type']]

# Display
print("\n" + "="*70)
print("TABLE 6: FEATURE IMPORTANCE RANKINGS")
print("="*70)
print(importance_df.to_string(index=False))

In [ ]:
print("Generating Figure 4: Feature Importance...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_eng, y_train_eng)

importance_df = pd.DataFrame({
    'Feature': X_train_eng.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

top_features = importance_df.tail(14)

original_features = ['TEMPERATURE', 'PH', 'OXYGEN', 'CONDUCTIVITY', 'TURB']
colors = ['#3498db' if feat in original_features else '#e67e22' for feat in top_features['Feature']]

plt.figure(figsize=(10, 8))
bars = plt.barh(top_features['Feature'], top_features['Importance'], color=colors, edgecolor='black', linewidth=1)

plt.xlabel('Importance Score', fontsize=14, fontweight='bold')
plt.ylabel('Feature', fontsize=14, fontweight='bold')
plt.title('Figure 4: Feature Importance Rankings for Cyanobacterial Bloom Prediction',
          fontsize=16, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#3498db', edgecolor='black', label='Original Sensor'),
                   Patch(facecolor='#e67e22', edgecolor='black', label='Engineered Feature')]
plt.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/figure4_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print(" Figure 4 saved!\n")

###ROC Curves
Visually compares model performance; curves closer to top-left = better

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay
import numpy as np

print("Generating ROC Curves for Individual Models...")

# Create figure with better spacing
fig, ax = plt.subplots(figsize=(10, 8))

# Color scheme for each model
colors = {
    'Logistic Regression': '#3498db',
    'Random Forest': '#2ecc71',
    'Gradient Boosting': '#e74c3c',
    'XGBoost': '#9b59b6',
    'Neural Network': '#f39c12'
}

# Plot ROC curves
RocCurveDisplay.from_estimator(best_lr, X_test_eng, y_test_eng, ax=ax,
                               name='Logistic Regression', color=colors['Logistic Regression'], lw=2.5)
RocCurveDisplay.from_estimator(best_rf, X_test_eng, y_test_eng, ax=ax,
                               name='Random Forest', color=colors['Random Forest'], lw=2.5)
RocCurveDisplay.from_estimator(best_gb, X_test_eng, y_test_eng, ax=ax,
                               name='Gradient Boosting', color=colors['Gradient Boosting'], lw=2.5)
RocCurveDisplay.from_estimator(best_xgb, X_test_eng, y_test_eng, ax=ax,
                               name='XGBoost', color=colors['XGBoost'], lw=2.5)
RocCurveDisplay.from_estimator(best_mlp, X_test_eng, y_test_eng, ax=ax,
                               name='Neural Network', color=colors['Neural Network'], lw=2.5)

# Add chance level line
ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance (AUC = 0.50)', alpha=0.6)

# Styling
ax.set_title('ROC Curves: Model Performance Comparison',
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('False Positive Rate', fontsize=14, fontweight='bold', labelpad=10)
ax.set_ylabel('True Positive Rate', fontsize=14, fontweight='bold', labelpad=10)

# Legend
ax.legend(loc='lower right', fontsize=11, framealpha=0.95,
          edgecolor='black', fancybox=True)

# Grid
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
ax.set_axisbelow(True)

# Clean spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Tick styling
ax.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/figure_roc_curves.png',
            dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print(" ROC Curves generated and saved!")

###ROC-AUC Model Comparison
Easy visual comparison of which model performed best

In [ ]:
print("Generating Figure 2: Model Performance Bar Chart...")

# Model scores from your results
model_scores = {
    'Logistic\nRegression': 0.7604,
    'Random\nForest': 0.7487,
    'Gradient\nBoosting': 0.7608,
    'Neural\nNetwork': 0.7446,
    'XGBoost': 0.7649,
    'Best\nEnsemble': 0.7723
}

models = list(model_scores.keys())
roc_scores = list(model_scores.values())

# Color scheme: blue for individual models, red for best ensemble
colors = ['#3498db', '#3498db', '#3498db', '#3498db', '#3498db', '#e74c3c']

# Create figure with better spacing
fig, ax = plt.subplots(figsize=(14, 8))

# Create bars with more width spacing
x_pos = np.arange(len(models))
bars = ax.bar(x_pos, roc_scores, width=0.6, color=colors,
              edgecolor='black', linewidth=1.5, alpha=0.85)

# Add value labels on top of bars
for i, (bar, score) in enumerate(zip(bars, roc_scores)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.001,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=13, fontweight='bold')

# Styling
ax.set_ylabel('ROC-AUC Score', fontsize=15, fontweight='bold', labelpad=10)
ax.set_xlabel('Machine Learning Model', fontsize=15, fontweight='bold', labelpad=10)
ax.set_title('Model Performance Comparison\nROC-AUC Scores for Cyanobacterial Bloom Prediction',
             fontsize=17, fontweight='bold', pad=20)

# Set x-axis
ax.set_xticks(x_pos)
ax.set_xticklabels(models, fontsize=12, fontweight='500')

# Y-axis improvements
ax.set_ylim(0.70, 0.80)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.01))
ax.tick_params(axis='y', labelsize=11)

# Grid for readability
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.7)
ax.set_axisbelow(True)

# Add horizontal line at 0.75 for reference
ax.axhline(y=0.75, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='0.75 threshold')

# Remove top and right spines for cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Tighter layout
plt.tight_layout()

# Save with high quality
plt.savefig('/content/drive/MyDrive/figure2_model_comparison.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(" Figure 2 saved!\n")

###Feature Distributions (Bloom vs. No-Bloom)
Shows which sensors separate blooms from non-blooms visually

In [ ]:
print("Generating Figure 3: Feature Distributions...")
import seaborn as sns

df = pd.read_csv('/content/drive/MyDrive/cyano_classification_data_threshold_10_with_turb.csv')

features = ['TEMPERATURE', 'PH', 'CONDUCTIVITY', 'TURB']
feature_labels = ['Temperature (°C)', 'pH', 'Conductivity (μS/cm)', 'Turbidity (NTU)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

no_bloom_color = '#3498db'
bloom_color = '#e74c3c'

for i, (feature, label) in enumerate(zip(features, feature_labels)):
    ax = axes[i]

    # Use seaborn for smoother distributions
    sns.histplot(data=df[df['CYANO_BLOOM'] == 0], x=feature,
                 ax=ax, kde=True, stat='density', bins=25,
                 color=no_bloom_color, alpha=0.5,
                 edgecolor='white', linewidth=0.5, label='No Bloom')

    sns.histplot(data=df[df['CYANO_BLOOM'] == 1], x=feature,
                 ax=ax, kde=True, stat='density', bins=25,
                 color=bloom_color, alpha=0.5,
                 edgecolor='white', linewidth=0.5, label='Bloom Present')

    # Calculate means
    no_bloom_mean = df[df['CYANO_BLOOM'] == 0][feature].mean()
    bloom_mean = df[df['CYANO_BLOOM'] == 1][feature].mean()

    # Plot mean lines
    ax.axvline(no_bloom_mean, color=no_bloom_color, linestyle='--',
               linewidth=2.5, alpha=0.9)
    ax.axvline(bloom_mean, color=bloom_color, linestyle='--',
               linewidth=2.5, alpha=0.9)

    # Add text box with means
    textstr = f'No Bloom: {no_bloom_mean:.2f}\nBloom: {bloom_mean:.2f}'
    props = dict(boxstyle='round', facecolor='white', edgecolor='gray', alpha=0.8)
    ax.text(0.97, 0.97, textstr, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', horizontalalignment='right', bbox=props)

    # Styling
    ax.set_title(label, fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('')
    ax.set_ylabel('Density', fontsize=12, labelpad=8)
    ax.legend(fontsize=10, loc='upper left', framealpha=0.95)
    ax.grid(alpha=0.2, linestyle='--', linewidth=0.6, axis='y')
    ax.set_axisbelow(True)

    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)

plt.suptitle('Feature Distributions: Bloom vs. No-Bloom Conditions',
             fontsize=17, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/figure3_feature_distributions.png',
            dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(" Figure 3 saved!\n")


###Confusion Matrix (for Best Model)
Shows what kinds of errors the best model makes

In [ ]:
print("Generating Confusion Matrix for the Best Model...")

# Get predictions
optimal_threshold = 0.35
y_pred_probs = final_model.predict_proba(X_test_eng)[:, 1]
y_pred_best = (y_pred_probs >= optimal_threshold).astype(int)

# Confusion matrix
cm_best = confusion_matrix(y_test_eng, y_pred_best)
tn, fp, fn, tp = cm_best.ravel()

# Create figure
fig, ax = plt.subplots(figsize=(8, 7))

# Custom annotations with labels
labels = np.array([
    [f'TN\n{tn}\n\nCorrectly\nNo Bloom', f'FP\n{fp}\n\nFalse\nAlarm'],
    [f'FN\n{fn}\n\nMissed\nBloom', f'TP\n{tp}\n\nCorrectly\nBloom']
])

# Heatmap
sns.heatmap(cm_best, annot=labels, fmt='',
            cmap='RdYlGn_r', cbar=False,
            square=True, linewidths=4, linecolor='white',
            annot_kws={'size': 14, 'weight': 'bold'},
            vmin=0, vmax=cm_best.max(),
            ax=ax)

# Labels and title
ax.set_ylabel('Actual', fontsize=17, fontweight='bold', labelpad=15)
ax.set_xlabel('Predicted', fontsize=17, fontweight='bold', labelpad=15)
ax.set_title(f'Confusion Matrix: {best_method} Ensemble\n(Threshold = {optimal_threshold})',
             fontsize=19, fontweight='bold', pad=20)

# Tick labels
ax.set_yticklabels(['No Bloom', 'Bloom'], fontsize=15, rotation=0, va='center')
ax.set_xticklabels(['No Bloom', 'Bloom'], fontsize=15, rotation=0, ha='center')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/figure5_confusion_matrix.png',
            dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print("Confusion Matrix generated!")

# Print metrics separately
print(f"\n{'='*50}")
print(f"Confusion Matrix Breakdown:")
print(f"{'='*50}")
print(f"True Negatives (TN):   {tn:3d}")
print(f"False Positives (FP):  {fp:3d}")
print(f"False Negatives (FN):  {fn:3d}")
print(f"True Positives (TP):   {tp:3d}")
print(f"\nFP:FN Ratio: {fp/fn:.2f}")
print(f"{'='*50}\n")

###Correlation Heatmap
Visually shows which features correlate with blooms and each other

In [ ]:
print("Generating Correlation Heatmap...")

df_eng = pd.read_csv('/content/drive/MyDrive/cyano_engineered.csv')
correlation_df = df_eng.drop(columns=['UID', 'CYANO_PCT'], errors='ignore')

corr_matrix = correlation_df.corr()

# Create figure
fig, ax = plt.subplots(figsize=(14, 12))

# Heatmap with cleaner styling
sns.heatmap(corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='RdBu_r',  # Red-Blue diverging
            center=0,
            square=True,
            linewidths=2,
            linecolor='white',
            cbar_kws={"shrink": 0.8, "label": "Pearson Correlation"},
            annot_kws={'size': 8},
            vmin=-1, vmax=1,
            ax=ax)

# Title
ax.set_title('Correlation Matrix: All Features',
             fontsize=18, fontweight='bold', pad=20)

# Labels
plt.xticks(rotation=45, ha='right', fontsize=10, fontweight='500')
plt.yticks(rotation=0, fontsize=10, fontweight='500')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/figure6_correlation_heatmap.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(" Heatmap saved!\n")

# Clean summary
print("="*50)
print("TOP 5 CORRELATIONS WITH BLOOM:")
print("="*50)
cyano_corr = correlation_df.corr()['CYANO_BLOOM'].sort_values(ascending=False)
for i, (feat, corr) in enumerate(cyano_corr.head(6).items(), 1):
    if feat != 'CYANO_BLOOM':
        print(f"{i}. {feat:25s} {corr:+.3f}")
print("="*50 + "\n")